# Aggregation and Grouping

## Quick notes
### What this covers
- Aggregate functions (COUNT, SUM, AVG, MIN, MAX, TOTAL, GROUP_CONCAT) that collapse many rows into one value.
- GROUP BY, HAVING, conditional aggregation (FILTER and CASE), subtotals, and grouping by date parts.
- How NULLs, integers and joins quietly change aggregate results.
### Key points
- An aggregate turns a set of rows into one value. `GROUP BY` makes one value per bucket.
- `WHERE` filters rows before grouping. `HAVING` filters groups after aggregating.
- `COUNT(*)` counts rows. `COUNT(col)` counts non-NULL values. `COUNT(DISTINCT col)` counts unique non-NULL values.
- Aggregates ignore NULLs (except `COUNT(*)`).
- Conditional aggregation (`SUM(CASE WHEN ...)` or `FILTER (WHERE ...)`) builds pivot-style reports in one pass.
### Gotchas (the part worth revising twice)
- `AVG` ignores NULLs, so it divides by fewer rows than you may expect.
- `SUM` of zero rows (or all NULLs) is NULL, not 0. `TOTAL` returns 0.0. Use `COALESCE(SUM(x), 0)`.
- Integer division: `SUM(a) / COUNT(*)` on integers truncates. Multiply by `1.0` first.
- Aggregating after a one-to-many join double counts the "one" side (fan-out trap).
- SQLite lets you select non-grouped, non-aggregated columns. The value is arbitrary, except with a lone `MIN`/`MAX` where it comes from the min/max row. Other databases raise an error.
- `SQLite` has no `ROLLUP` or `CUBE`; build subtotals with `UNION ALL`.
### When it matters
- Every report and dashboard: totals per month, headcount per department, top customers.
- Data checks: duplicates (`GROUP BY ... HAVING COUNT(*) > 1`).
### One-line recap example
- `SELECT dept, count(*), avg(salary) FROM employees GROUP BY dept HAVING count(*) > 1;` means "per department, headcount and average pay, only for departments with more than one person".

## 01. Seed data
- Employees (with some NULL bonuses), a small sales log with fixed dates, and orders with line items.
- Data is small enough to check results by eye.

In [1]:
# setup: an in-memory SQLite database and a %%sql cell magic that prints result tables
import sqlite3
from IPython.core.magic import register_cell_magic

con = sqlite3.connect(":memory:", isolation_level=None)   # autocommit, so BEGIN/COMMIT work as written
con.execute("PRAGMA foreign_keys = ON")

def _table(cols, rows):
    cells = [[("NULL" if v is None else str(v)) for v in r] for r in rows]
    widths = [max([len(c)] + [len(r[i]) for r in cells]) for i, c in enumerate(cols)]
    line = lambda vals: " | ".join(v.ljust(w) for v, w in zip(vals, widths))
    print(line(cols)); print("-+-".join("-" * w for w in widths))
    for r in cells: print(line(r))
    print(f"({len(rows)} row{'s' if len(rows) != 1 else ''})\n")

def _statements(text):
    """Split a cell into statements (handles several per line, triggers, quoted semicolons)."""
    out, start = [], 0
    for i, ch in enumerate(text):
        if ch == ";" and sqlite3.complete_statement(text[start:i + 1]):
            out.append(text[start:i + 1].strip()); start = i + 1
    if text[start:].strip():
        out.append(text[start:].strip())
    return out

@register_cell_magic
def sql(line, cell):
    """Run every statement in the cell; print rows for statements that return them."""
    expect_error = "expect_error" in line
    for stmt in _statements(cell):
        try:
            cur = con.execute(stmt)
            if cur.description:
                _table([d[0] for d in cur.description], cur.fetchall())
        except sqlite3.Error as exc:
            if not expect_error:
                raise
            print(f"ERROR: {exc}\n")

In [2]:
%%sql
CREATE TABLE employees (
    emp_id INTEGER PRIMARY KEY, first_name TEXT NOT NULL, dept TEXT NOT NULL,
    region TEXT NOT NULL, salary INTEGER NOT NULL, bonus INTEGER
);
INSERT INTO employees VALUES
    (1,'Alice','Engineering','West',95000,5000),
    (2,'Bob','Engineering','East',72000,NULL),
    (3,'Carla','Sales','West',68000,3000),
    (4,'Dev','Sales','West',61000,NULL),
    (5,'Eve','Sales','East',58000,2000),
    (6,'Finn','Marketing','East',75000,NULL);
CREATE TABLE sales (
    sale_id INTEGER PRIMARY KEY, sale_date TEXT NOT NULL, region TEXT NOT NULL,
    product TEXT NOT NULL, qty INTEGER NOT NULL, amount INTEGER NOT NULL
);
INSERT INTO sales VALUES
    (1,'2025-01-05','West','Widget',3,30),(2,'2025-01-20','East','Gadget',1,25),
    (3,'2025-02-02','West','Gadget',2,50),(4,'2025-02-14','West','Widget',5,50),
    (5,'2025-02-28','East','Widget',4,40),(6,'2025-03-03','East','Gadget',2,50),
    (7,'2025-03-15','West','Widget',1,10),(8,'2025-03-31','West','Gizmo',6,90);
SELECT * FROM employees ORDER BY emp_id;

emp_id | first_name | dept        | region | salary | bonus
-------+------------+-------------+--------+--------+------
1      | Alice      | Engineering | West   | 95000  | 5000 
2      | Bob        | Engineering | East   | 72000  | NULL 
3      | Carla      | Sales       | West   | 68000  | 3000 
4      | Dev        | Sales       | West   | 61000  | NULL 
5      | Eve        | Sales       | East   | 58000  | 2000 
6      | Finn       | Marketing   | East   | 75000  | NULL 
(6 rows)



## 02. The basic aggregates
- `COUNT`, `SUM`, `AVG`, `MIN`, `MAX` over the whole table give a single row.
- `MIN`/`MAX` also work on text (alphabetical) and dates stored as ISO text.

In [3]:
%%sql
SELECT count(*) AS n, sum(salary) AS total, avg(salary) AS average,
       min(salary) AS lowest, max(salary) AS highest,
       min(first_name) AS first_alpha, max(first_name) AS last_alpha
FROM employees;

n | total  | average | lowest | highest | first_alpha | last_alpha
--+--------+---------+--------+---------+-------------+-----------
6 | 429000 | 71500.0 | 58000  | 95000   | Alice       | Finn      
(1 row)



## 03. Aggregates and NULLs
- Aggregates skip NULL values. Only `COUNT(*)` counts every row.
- Three employees have no bonus, so `COUNT(bonus)` is 3, not 6.
- `AVG(bonus)` is 10000/3, not 10000/6. If "no bonus" should mean 0, say so with `COALESCE`.

In [4]:
%%sql
SELECT count(*) AS all_rows, count(bonus) AS non_null_bonus,
       sum(bonus) AS total_bonus,
       avg(bonus) AS avg_ignoring_null,
       avg(coalesce(bonus, 0)) AS avg_null_as_zero
FROM employees;

all_rows | non_null_bonus | total_bonus | avg_ignoring_null  | avg_null_as_zero  
---------+----------------+-------------+--------------------+-------------------
6        | 3              | 10000       | 3333.3333333333335 | 1666.6666666666667
(1 row)



## 04. DISTINCT inside aggregates
- `COUNT(DISTINCT x)` counts unique non-NULL values. Also works with `SUM(DISTINCT x)` and `AVG(DISTINCT x)`.
- Distinct sum is rarely what you want: two equal salaries would be counted once.

In [5]:
%%sql
SELECT count(*) AS employees, count(DISTINCT dept) AS departments, count(DISTINCT region) AS regions,
       sum(DISTINCT salary) AS sum_distinct_salaries
FROM employees;

employees | departments | regions | sum_distinct_salaries
----------+-------------+---------+----------------------
6         | 3           | 2       | 429000               
(1 row)



## 05. GROUP BY: one or many columns
- Rows with the same value(s) form a group; aggregates run per group.
- Group by two columns for finer buckets: each (dept, region) pair is its own group.
- Add `ORDER BY`; group output order is otherwise not guaranteed.

In [6]:
%%sql
SELECT dept, count(*) AS headcount, avg(salary) AS avg_salary
FROM employees
GROUP BY dept
ORDER BY dept;
SELECT dept, region, count(*) AS headcount
FROM employees
GROUP BY dept, region
ORDER BY dept, region;

dept        | headcount | avg_salary        
------------+-----------+-------------------
Engineering | 2         | 83500.0           
Marketing   | 1         | 75000.0           
Sales       | 3         | 62333.333333333336
(3 rows)

dept        | region | headcount
------------+--------+----------
Engineering | East   | 1        
Engineering | West   | 1        
Marketing   | East   | 1        
Sales       | East   | 1        
Sales       | West   | 2        
(5 rows)



## 06. GROUP BY expressions, ordinals and aliases
- You can group by an expression, e.g. a salary band computed with `CASE`.
- SQLite also lets you `GROUP BY 1` (position in the SELECT list) or by a SELECT alias. Handy for quick work, but not portable and breaks when you reorder columns.
- Prefer repeating the expression or using a CTE in shared code.

In [7]:
%%sql
SELECT CASE WHEN salary >= 70000 THEN 'high' ELSE 'standard' END AS pay_band,
       count(*) AS people
FROM employees
GROUP BY pay_band
ORDER BY pay_band;
SELECT dept, count(*) AS people FROM employees GROUP BY 1 ORDER BY 1;

pay_band | people
---------+-------
high     | 3     
standard | 3     
(2 rows)

dept        | people
------------+-------
Engineering | 2     
Marketing   | 1     
Sales       | 3     
(3 rows)



## 07. WHERE vs HAVING
- `WHERE` filters individual rows before grouping. It cannot use aggregates.
- `HAVING` filters groups after aggregating. It can use aggregates.
- Rule of thumb: filter by raw columns in `WHERE` (cheaper), by aggregate results in `HAVING`.

In [8]:
%%sql
SELECT dept, count(*) AS headcount, avg(salary) AS avg_salary
FROM employees
WHERE region = 'West'
GROUP BY dept
HAVING count(*) >= 1
ORDER BY dept;
SELECT dept, count(*) AS headcount
FROM employees
GROUP BY dept
HAVING count(*) > 1
ORDER BY dept;

dept        | headcount | avg_salary
------------+-----------+-----------
Engineering | 1         | 95000.0   
Sales       | 2         | 64500.0   
(2 rows)

dept        | headcount
------------+----------
Engineering | 2        
Sales       | 3        
(2 rows)



In [9]:
%%sql expect_error
-- mistake: aggregate in WHERE
SELECT dept FROM employees WHERE count(*) > 1 GROUP BY dept;

ERROR: misuse of aggregate: count()



## 08. Bare columns and the SQLite min/max quirk
- Selecting a column that is neither grouped nor aggregated is an error in most databases. SQLite allows it.
- With plain columns the value comes from an arbitrary row of the group: do not trust it.
- Special rule: when the query has exactly one `MIN` or `MAX`, bare columns come from that min/max row. Handy for "who earns the most in each department".
- Portable alternatives: window functions (see 03-Window-Functions/01-Window-Functions) or a join to the aggregated subquery.

In [10]:
%%sql
SELECT dept, first_name, max(salary) AS top_salary
FROM employees
GROUP BY dept
ORDER BY dept;

dept        | first_name | top_salary
------------+------------+-----------
Engineering | Alice      | 95000     
Marketing   | Finn       | 75000     
Sales       | Carla      | 68000     
(3 rows)



## 09. FILTER (WHERE ...) on aggregates
- `agg(...) FILTER (WHERE cond)` aggregates only rows meeting `cond`. Available in SQLite 3.30+.
- Reads better than `SUM(CASE ...)`: several differently filtered numbers in one query.

In [11]:
%%sql
SELECT dept,
       count(*) AS everyone,
       count(*) FILTER (WHERE region = 'West') AS west_people,
       avg(salary) FILTER (WHERE region = 'East') AS east_avg_salary
FROM employees
GROUP BY dept
ORDER BY dept;

dept        | everyone | west_people | east_avg_salary
------------+----------+-------------+----------------
Engineering | 2        | 1           | 72000.0        
Marketing   | 1        | 0           | 75000.0        
Sales       | 3        | 2           | 58000.0        
(3 rows)



## 10. Conditional aggregation with CASE
- The older and portable way: `SUM(CASE WHEN cond THEN x ELSE 0 END)`.
- Use it to pivot rows into columns: one column per month, region or status.
- To count with CASE, return 1 or NULL (`COUNT(CASE WHEN c THEN 1 END)`), since COUNT skips NULLs.

In [12]:
%%sql
SELECT product,
       sum(CASE WHEN region = 'West' THEN amount ELSE 0 END) AS west_amount,
       sum(CASE WHEN region = 'East' THEN amount ELSE 0 END) AS east_amount,
       count(CASE WHEN amount >= 50 THEN 1 END) AS big_sales
FROM sales
GROUP BY product
ORDER BY product;

product | west_amount | east_amount | big_sales
--------+-------------+-------------+----------
Gadget  | 50          | 75          | 2        
Gizmo   | 90          | 0           | 1        
Widget  | 90          | 40          | 1        
(3 rows)



## 11. GROUP_CONCAT: order, separator, DISTINCT
- Joins values of a group into one string. Default separator is a comma.
- Pass a second argument for the separator. With `DISTINCT` you cannot also give a separator in SQLite.
- Order inside the string is not guaranteed unless you aggregate from an ordered subquery (works in practice) or use `ORDER BY` inside the aggregate (SQLite 3.44+).

In [13]:
%%sql
SELECT dept, group_concat(first_name, ', ' ORDER BY first_name) AS people
FROM employees
GROUP BY dept
ORDER BY dept;
SELECT region, group_concat(DISTINCT product) AS products_sold
FROM (SELECT region, product FROM sales ORDER BY product)
GROUP BY region
ORDER BY region;

dept        | people         
------------+----------------
Engineering | Alice, Bob     
Marketing   | Finn           
Sales       | Carla, Dev, Eve
(3 rows)

region | products_sold      
-------+--------------------
East   | Gadget,Widget      
West   | Gadget,Gizmo,Widget
(2 rows)



## 12. Subtotals and grand total without ROLLUP
- SQLite has no `ROLLUP`, `CUBE` or `GROUPING SETS`.
- Emulate with several GROUP BY queries stitched together by `UNION ALL` (use `ALL`: no needless de-duplication).
- Use NULL (or a label) for the "all" level and add a sort key so the totals appear last.
- See 02-Querying/04-Set-Operations for UNION ALL rules.

In [14]:
%%sql
SELECT * FROM (
SELECT region, product, sum(amount) AS amount, 0 AS lvl FROM sales GROUP BY region, product
UNION ALL
SELECT region, '(all products)', sum(amount), 1 FROM sales GROUP BY region
UNION ALL
SELECT '(all regions)', '(all products)', sum(amount), 2 FROM sales
)
ORDER BY region = '(all regions)', region, lvl, product;

region        | product        | amount | lvl
--------------+----------------+--------+----
East          | Gadget         | 75     | 0  
East          | Widget         | 40     | 0  
East          | (all products) | 115    | 1  
West          | Gadget         | 50     | 0  
West          | Gizmo          | 90     | 0  
West          | Widget         | 90     | 0  
West          | (all products) | 230    | 1  
(all regions) | (all products) | 345    | 2  
(8 rows)



## 13. Precision gotchas: integer AVG, empty SUM, TOTAL
- `AVG` always returns a float, but your own `SUM(x) / COUNT(*)` on integers truncates.
- `SUM` over zero rows (or only NULLs) is NULL; `TOTAL` returns 0.0 instead; `COUNT` returns 0.
- Safe idiom: `COALESCE(SUM(x), 0)`.
- Floats add imprecisely; `ROUND` for display, and store money as integer cents.

In [15]:
%%sql
SELECT sum(salary) / count(*) AS int_division,
       sum(salary) * 1.0 / count(*) AS float_division,
       avg(salary) AS avg_builtin
FROM employees;
SELECT sum(salary) AS sum_empty, total(salary) AS total_empty,
       count(*) AS count_empty, max(salary) AS max_empty,
       coalesce(sum(salary), 0) AS safe_sum
FROM employees WHERE dept = 'Nonexistent';

int_division | float_division | avg_builtin
-------------+----------------+------------
71500        | 71500.0        | 71500.0    
(1 row)

sum_empty | total_empty | count_empty | max_empty | safe_sum
----------+-------------+-------------+-----------+---------
NULL      | 0.0         | 0           | NULL      | 0       
(1 row)



## 14. Group by date parts with strftime
- Dates stored as ISO text (`YYYY-MM-DD`) can be bucketed with `strftime`: `'%Y-%m'` for month, `'%Y'` year, `'%w'` weekday.
- Group by the expression, not the raw date, or every day becomes its own group.
- Months with no rows are absent from the result. To show them, join to a calendar table (see 02-Querying/01-Joins, CROSS JOIN).

In [16]:
%%sql
SELECT strftime('%Y-%m', sale_date) AS month, count(*) AS n_sales, sum(amount) AS revenue
FROM sales
GROUP BY month
ORDER BY month;
SELECT strftime('%Y', sale_date) AS yr, strftime('%m', sale_date) AS mo,
       sum(qty) AS units
FROM sales
GROUP BY yr, mo
ORDER BY yr, mo;

month   | n_sales | revenue
--------+---------+--------
2025-01 | 2       | 55     
2025-02 | 3       | 140    
2025-03 | 3       | 150    
(3 rows)

yr   | mo | units
-----+----+------
2025 | 01 | 4    
2025 | 02 | 11   
2025 | 03 | 9    
(3 rows)



## 15. The fan-out trap: aggregating over a join
- Orders have a header-level `shipping_fee`. Joining to line items repeats each order once per item.
- Summing `shipping_fee` after the join counts it once per item: wrong total.
- Fix: aggregate each side separately in subqueries/CTEs, then join the results. See 02-Querying/01-Joins (join explosion).

In [17]:
%%sql
CREATE TABLE orders (order_id INTEGER PRIMARY KEY, customer TEXT NOT NULL, shipping_fee INTEGER NOT NULL);
CREATE TABLE order_items (order_id INTEGER NOT NULL, sku TEXT NOT NULL, qty INTEGER NOT NULL, price INTEGER NOT NULL);
INSERT INTO orders VALUES (1,'Ann',5),(2,'Ben',10),(3,'Ann',5);
INSERT INTO order_items VALUES (1,'A',2,10),(1,'B',1,20),(1,'C',1,5),(2,'A',1,10),(3,'B',3,20);
SELECT sum(shipping_fee) AS correct_shipping FROM orders;
SELECT sum(o.shipping_fee) AS inflated_shipping, sum(i.qty * i.price) AS item_revenue
FROM orders o JOIN order_items i ON i.order_id = o.order_id;

correct_shipping
----------------
20              
(1 row)

inflated_shipping | item_revenue
------------------+-------------
30                | 115         
(1 row)



In [18]:
%%sql
-- good: collapse items to one row per order first
WITH item_totals AS (
    SELECT order_id, sum(qty * price) AS items_total FROM order_items GROUP BY order_id
)
SELECT o.customer, sum(o.shipping_fee) AS shipping, sum(t.items_total) AS items
FROM orders o JOIN item_totals t ON t.order_id = o.order_id
GROUP BY o.customer
ORDER BY o.customer;

customer | shipping | items
---------+----------+------
Ann      | 10       | 105  
Ben      | 10       | 10   
(2 rows)



## 16. NULL is a group, and empty tables
- `GROUP BY` puts all NULLs into one group (unlike `=`, which treats NULLs as different).
- A whole-table aggregate over an empty set still returns one row; a `GROUP BY` over an empty set returns zero rows.

In [19]:
%%sql
SELECT bonus IS NULL AS no_bonus, count(*) AS people
FROM employees
GROUP BY bonus IS NULL
ORDER BY no_bonus;
SELECT dept, count(*) FROM employees WHERE 1 = 0 GROUP BY dept;
SELECT count(*) AS n FROM employees WHERE 1 = 0;

no_bonus | people
---------+-------
0        | 3     
1        | 3     
(2 rows)

dept | count(*)
-----+---------
(0 rows)

n
-
0
(1 row)



## 17. Finding duplicates and top groups
- `GROUP BY cols HAVING COUNT(*) > 1` is the standard duplicate detector.
- Sort aggregates with `ORDER BY` on the aggregate (or alias) plus `LIMIT` for "top N".
- Add a tie-breaker column to `ORDER BY` so the result is deterministic.

In [20]:
%%sql
SELECT customer, count(*) AS n_orders FROM orders GROUP BY customer HAVING count(*) > 1 ORDER BY customer;
SELECT product, sum(amount) AS revenue
FROM sales
GROUP BY product
ORDER BY revenue DESC, product
LIMIT 2;

customer | n_orders
---------+---------
Ann      | 2       
(1 row)

product | revenue
--------+--------
Widget  | 130    
Gadget  | 125    
(2 rows)

